In [2]:
# # pip install pandas-gbq google-cloud-bigquery
# import pandas as pd 
# from google.cloud import bigquery

# def fetch_gdelt_sentiment(start_date: str, end_date: str) -> pd.DataFrame:
#     """
#     GDELT에서 금융 관련 기사 감성 점수 일별 집계
#     - Tone: 음수(부정) ~ 양수(긍정), 보통 -10 ~ +10
#     - 완전 무료, 10년치 가능
#     """
#     client = bigquery.Client()
#     query = f"""
#     SELECT
#         PARSE_DATE('%Y%m%d', CAST(DATE AS STRING)) AS date,
#         AVG(V2Tone_1) AS avg_sentiment,   -- 전체 감성 평균
#         STDDEV(V2Tone_1) AS std_sentiment, -- 변동성 (불확실성 프록시)
#         COUNT(*) AS article_count
#     FROM `gdelt-bq.gdeltv2.gkg`
#     WHERE DATE BETWEEN {start_date.replace('-','')} AND {end_date.replace('-','')}
#       AND (
#         Themes LIKE '%ECON_STOCKMARKET%'
#         OR Themes LIKE '%ECON_FRBRESERVE%'
#         OR Themes LIKE '%ECON_INFLATION%'
#       )
#       AND SourceCommonName IN (
#         'reuters.com', 'cnbc.com', 'wsj.com',
#         'ft.com', 'bloomberg.com'
#       )
#     GROUP BY date
#     ORDER BY date
#     """
#     return client.query(query).to_dataframe()

# gdelt_df = fetch_gdelt_sentiment('2015-01-01', '2025-01-01')

In [3]:
# # ============================================================
# # 전체 뉴스 감성 분석 파이프라인
# # ============================================================
# # pip install pandas-gbq google-cloud-bigquery yfinance
# #             newsapi-python transformers torch numpy pandas

# import numpy as np
# import pandas as pd
# from datetime import datetime, timedelta
# from transformers import pipeline
# import yfinance as yf
# from newsapi import NewsApiClient

# # ============================================================
# # 0. 공통 모델 초기화 (전역 1회만 로드)
# # FinBERT는 딥러닝 모델이라 로딩에 수십초가 걸림! -> 1회만 로드하고 모든 함수가 공유해서 사용하면 더 효율적
# # FinBERT는 금융 뉴스로 파인튜닝된 모델이라 금융 맥락을 정확히 파악
# # ============================================================
# print("FinBERT 모델 로딩 중...")
# sentiment_pipe = pipeline(
#     "text-classification",
#     model="ProsusAI/finbert",
#     top_k=None          # positive/negative/neutral 3개 확률 모두 반환 (기본값top_k=1: 가장 높은 라벨 하나만 반환하여 점수 계산 불가)
# )
# print("모델 로딩 완료")


# # ============================================================
# # 공통 유틸: 텍스트 → 감성 점수 (-1 ~ +1)
# # ============================================================
# def finbert_score(text: str) -> float:
#     """
#     FinBERT로 단일 텍스트 감성 점수 계산
#     반환값: positive 확률 - negative 확률 → -1 ~ +1

#     [반환값 해석]
#     +1.0에 가까울수록 → 매우 긍정적 뉴스 (예: "Apple beats earnings")
#      0.0에 가까울수록 → 중립적 뉴스    (예: "Fed holds rates steady")
#     -1.0에 가까울수록 → 매우 부정적 뉴스 (예: "Market crash fears grow")

#     [계산 방식]
#     FinBERT가 반환하는 positive 확률 - negative 확률
#     예: positive=0.8, negative=0.1 → 점수 = +0.7
#         positive=0.1, negative=0.7 → 점수 = -0.6
#     """
#     if not text or len(text.strip()) < 5:
#         return 0.0
#     results = sentiment_pipe(text[:512])[0]   # 토큰 512 제한
#     score_map = {r['label']: r['score'] for r in results}
#     return score_map.get('positive', 0.0) - score_map.get('negative', 0.0)

# # ============================================================
# # 공통 유틸 2: 감성 점수 리스트 → 블랙-리터만 Ω
# # ============================================================
# def scores_to_omega(scores: list) -> float:
#     """
#     감성 점수 리스트 → 블랙-리터만 확신도 Ω

#     [Ω(Omega)란?]
#     블랙-리터만에서 투자자의 뷰(전망)를 얼마나 강하게 믿는지를 나타내는 값
#     Ω가 낮을수록 → 뷰를 강하게 믿음 → 포트폴리오 비중에 크게 반영
#     Ω가 높을수록 → 뷰를 약하게 믿음 → 시장 균형 비중에 가깝게 유지

#     [Ω 결정 기준]
#     두 가지 조건을 동시에 봅니다:
#       1. 기사 수(volume)    : 많을수록 신뢰도 상승
#       2. 감성 일관성        : 여러 기사가 같은 방향을 가리킬수록 신뢰도 상승

#     [예시]
#     XLK 관련 기사 15건이 모두 긍정적 → Ω=0.01 (강한 확신 → 비중 크게 확대)
#     XLE 관련 기사 3건이 방향 제각각  → Ω=0.10 (약한 확신 → 비중 소폭만 조정)
#     """
#     if not scores:
#         return 0.10     # 기사 없으면 확신도 최저

#     volume      = len(scores)
#     consistency = 1.0 - np.std(scores)     # std 작을수록 일관성 높음

#     if volume >= 10 and consistency > 0.7:
#         return 0.01     # 강한 확신: 기사 많고 방향 일치
#     elif volume >= 5 and consistency > 0.5:
#         return 0.05     # 중간 확신
#     else:
#         return 0.10     # 약한 확신: 기사 적거나 방향 불일치


# # ============================================================
# # 1. GDELT — 백테스트용 10년치 시장 감성 (레짐 분류 기반)
# # ============================================================
# def fetch_gdelt_sentiment(start_date: str, end_date: str) -> pd.DataFrame:
#     """
#     GDELT BigQuery에서 금융 테마 기사의 일별 감성 집계
#     - avg_sentiment : 일별 평균 감성 (-10 ~ +10) → 레짐 분류(Bull/Bear/Crisis)의 핵심 입력값
#     - std_sentiment : 일별 감성 변동성 → 변동성 클수록 시장 의견 분분 → Ω 높게 설정
#     - article_count : 일별 기사 수 → 급등 시 시장 주목도 상승 신호로 활용 (시장 관심도 피처)
    
#     [V2Tone 컬럼 구조]
#     GDELT의 V2Tone은 콤마로 구분된 문자열
#     "3.5, 5.2, 1.7, ..." 형태에서 SPLIT(...)[OFFSET(0)]으로
#     첫 번째 값(전체 감성)만 추출합니다.

#     [사전 준비]
#       1. Google Cloud 프로젝트 생성 (무료)
#       2. BigQuery API 활성화
#       3. gcloud auth application-default login 실행
#     """
#     from google.cloud import bigquery
#     client = bigquery.Client()

#     # V2Tone 컬럼: "전체감성,긍정비율,부정비율,극성,..." 콤마 구분 문자열
#     # SPLIT(...)[OFFSET(0)] 으로 첫 번째 값(전체 감성)만 추출
#     query = f"""
#     SELECT
#         PARSE_DATE('%Y%m%d', CAST(DATE AS STRING))  AS date,
#         AVG(CAST(SPLIT(V2Tone, ',')[OFFSET(0)] AS FLOAT64)) AS avg_sentiment,
#         STDDEV(CAST(SPLIT(V2Tone, ',')[OFFSET(0)] AS FLOAT64)) AS std_sentiment,
#         COUNT(*) AS article_count
#     FROM `gdelt-bq.gdeltv2.gkg`
#     WHERE DATE BETWEEN {start_date.replace('-', '')}
#                    AND {end_date.replace('-', '')}
#       AND (
#         -- 주식시장, 연준, 인플레이션, 실업 관련 기사만 필터링
#         -- 이 4개 테마가 레짐 판별에 가장 유의미한 신호를 제공
#             Themes LIKE '%ECON_STOCKMARKET%' # 주식
#          OR Themes LIKE '%ECON_FRBRESERVE%' # 연준
#          OR Themes LIKE '%ECON_INFLATION%' # 인플레이션
#          OR Themes LIKE '%ECON_UNEMPLOYMENT%' # 실업
#       )
#         -- 노이즈 줄이기 위해 신뢰도 높은 5개 소스만 사용
#         -- 일반 블로그나 소셜미디어 기사 제외
#       AND SourceCommonName IN (
#             'reuters.com', 'cnbc.com', 'wsj.com',
#             'ft.com', 'bloomberg.com'
#       )
#     GROUP BY date
#     ORDER BY date
#     """
#     df = client.query(query).to_dataframe()
#     df['date'] = pd.to_datetime(df['date'])
#     df = df.set_index('date')
#     return df


# def classify_regime(gdelt_df: pd.DataFrame,
#                     vix_series: pd.Series,
#                     spread_series: pd.Series) -> pd.Series:
#     """
#     GDELT 감성 + VIX(공포 지수) + 하이일드 스프레드(신용 위험) → 레짐 분류
#     - Crisis  : 감성 급락 + VIX 급등
#     - Bear    : 감성 부정 + 스프레드 확대
#     - Bull    : 감성 긍정 + VIX 안정
#     - Neutral : 그 외
    
#     스트레스 테스트 구간:
#       평균 - 2σ 이하로 감성이 떨어지는 날짜를 자동 식별

#     [입력값]
#     gdelt_df      : fetch_gdelt_sentiment() 반환값
#     vix_series    : 기존 yf_data['^VIX'] (날짜 인덱스 맞춰야 함)
#     spread_series : 기존 fred_data['BAMLH0A0HYM2']
#     """
#     df = gdelt_df.copy()
#     df['VIX']    = vix_series
#     df['spread'] = spread_series

#     mean = df['avg_sentiment'].mean()
#     std  = df['avg_sentiment'].std()

#     def label(row):
#         if row['avg_sentiment'] < mean - 2 * std and row['VIX'] > 30:
#             return 'Crisis'
#         elif row['avg_sentiment'] < mean - std and row['spread'] > 5:
#             return 'Bear'
#         elif row['avg_sentiment'] > mean + std and row['VIX'] < 18:
#             return 'Bull'
#         else:
#             return 'Neutral'

#     regime      = df.apply(label, axis=1)
#     stress_days = df[df['avg_sentiment'] < mean - 2 * std].index   # 스트레스 테스트 구간 출력 (확인용)
#     print(f"스트레스 테스트 구간 {len(stress_days)}일 식별 완료")
#     print(stress_days[:5])   # 상위 5개 확인용

#     return regime


# # ============================================================
# # 2. yfinance 뉴스 — 백테스트 + 현재 모두 사용
# #    섹터/종목별 Q(전망값) 및 Ω(확신도) 계산
# # ============================================================
# def get_yf_sentiment(ticker_symbol: str) -> dict:
#     """
#     yfinance 내장 뉴스로 종목/섹터별 감성 점수와 확신도 계산
#     반환: {
#         'Q'    : 블랙-리터만 전망값 (감성 점수 평균),
#         'Omega': 블랙-리터만 확신도,
#         'count': 수집된 기사 수
#     }

#     [Q(전망값)란?]
#     "이 종목이 시장 대비 얼마나 더/덜 오를 것인가"에 대한 수치
#     Q > 0 → 긍정적 뉴스 → 시장 대비 초과수익 기대 → 비중 확대 신호
#     Q < 0 → 부정적 뉴스 → 시장 대비 하회 예상   → 비중 축소 신호

#     [Ω(확신도)란?]
#     Q를 얼마나 강하게 믿는지를 나타내는 값 (scores_to_omega 참고)

#     [반환값 예시]
#     XLK(기술주) 뉴스가 긍정적일 때:
#     {'Q': +0.65, 'Omega': 0.01, 'count': 12}
#     → "기술주에 강한 긍정 뷰, 높은 확신도 → 비중 크게 확대"

#     [한계]
#     yfinance .news는 최근 약 8~10개 헤드라인만 제공
#     과거 시점의 뉴스를 날짜별로 조회하는 것은 불가
#     → 백테스트에서는 현재 시점 뷰 참고용, GDELT가 메인
#     """
#     ticker = yf.Ticker(ticker_symbol)
#     news   = ticker.news or []

#     if not news:
#         return {'Q': 0.0, 'Omega': 0.10, 'count': 0}

#     scores = [finbert_score(n.get('title', '')) for n in news]

#     return {
#         'Q'    : float(np.mean(scores)),    # 전망값: 양수면 긍정 뷰
#         'Omega': scores_to_omega(scores),   # 확신도: 낮을수록 강한 확신
#         'count': len(scores)
#     }


# def get_portfolio_views(tickers: list) -> pd.DataFrame:
#     """
#     포트폴리오 전체 티커에 대해 Q, Ω 일괄 계산
#     블랙-리터만 모델 입력값으로 바로 사용 가능

#     [반환 DataFrame 구조]
#             Q       Omega  count
#     XLK  +0.65    0.01     12    ← 강한 긍정 뷰, 높은 확신도
#     XLE  -0.42    0.05      7    ← 부정 뷰, 중간 확신도
#     TLT  +0.31    0.10      3    ← 긍정 뷰지만 기사 적어 낮은 확신도
#     """
#     records = []
#     for ticker in tickers:
#         result = get_yf_sentiment(ticker)
#         records.append({'ticker': ticker, **result})
#         print(f"  {ticker:6s} | Q={result['Q']:+.3f} | Ω={result['Omega']:.2f} | 기사수={result['count']}")

#     return pd.DataFrame(records).set_index('ticker')


# # ============================================================
# # 3. NewsAPI — 현재 시점 실시간 리밸런싱 전용
# #    yfinance Ω를 고품질 외신 기준으로 보완
# # ============================================================
# newsapi = NewsApiClient(api_key='YOUR_API_KEY')

# def get_current_omega(keyword: str, days_back: int = 7) -> dict:
#     """
#     현재 시점 리밸런싱 시 NewsAPI로 Ω 보완
#     - 고품질 외신(Reuters, Bloomberg 등)만 필터링
#     - 기사량 + 감성 일관성으로 확신도 재계산
    
#     반환: {
#         'mean_score' : 평균 감성 점수,
#         'Omega'      : 확신도,
#         'volume'     : 기사 수
#     }

#     [반환값 예시]
#     keyword='Federal Reserve interest rate' 검색 시:
#     {'mean_score': -0.42, 'Omega': 0.01, 'volume': 18}
#     → "Fed 관련 뉴스 18건이 일관되게 부정적 → 채권 비중 확대 강한 확신"
#     """
#     from_date = (datetime.now() - timedelta(days=days_back)).strftime('%Y-%m-%d')

#     response = newsapi.get_everything(
#         q=keyword,
#         from_param=from_date,
#         language='en',
#         sort_by='relevancy',
#         sources='reuters,bloomberg,cnbc,the-wall-street-journal'
#     )
#     articles = response.get('articles', [])[:20]   # 상위 20개만 사용

#     if not articles:
#         return {'mean_score': 0.0, 'Omega': 0.10, 'volume': 0}

#     scores = [finbert_score(a.get('title', '')) for a in articles]

#     return {
#         'mean_score': float(np.mean(scores)),
#         'Omega'     : scores_to_omega(scores),
#         'volume'    : len(scores)
#     }


# # ============================================================
# # 4. 실행 예시
# # ============================================================
# if __name__ == "__main__":

#     # ── Step 1: GDELT 백테스트 데이터 수집 ──
#     # 결과: 날짜별 (avg_sentiment, std_sentiment, article_count) 테이블
#     print("\n[1] GDELT 감성 데이터 수집 중...")
#     gdelt_df = fetch_gdelt_sentiment('2015-01-01', '2025-01-01')
#     print(gdelt_df.head())

#     # ── Step 2: 레짐 분류 (yfinance에서 VIX, 스프레드 별도 수집 필요) ──
#     # 주의: vix_series, spread_series는 기존 yf_data/fred_data에서 가져옴
#     # gdelt_df와 날짜 인덱스를 맞춰야 join 가능
#     # regime_series = classify_regime(gdelt_df, vix_series, spread_series)
#     # master_df['regime'] = regime_series  → 포트폴리오 비중 조정 트리거

#     # ── Step 3: 섹터/종목별 Q, Ω 계산 ──
#     # 결과: 블랙-리터만 모델에 바로 입력 가능한 DataFrame
#     print("\n[2] 섹터/종목별 뉴스 감성 분석 중...")
#     SECTOR_ETF = ['XLK', 'XLF', 'XLE', 'XLV', 'XLY', 'XLP', 'XLI', 'XLU', 'XLRE', 'XLB', 'VOX']
#     STOCKS     = ['AAPL', 'MSFT', 'AMZN', 'GOOGL', 'JPM', 'JNJ', 'PG', 'XOM']

#     views_df = get_portfolio_views(SECTOR_ETF + STOCKS)
#     print("\n블랙-리터만 입력값 (Q, Ω):")
#     print(views_df)

#     # ── Step 4: 현재 시점 NewsAPI 확신도 보완 ──
#     # 매주 리밸런싱 시 아래 신호들을 참고하여 최종 비중 결정
#     print("\n[3] NewsAPI 매크로 확신도 계산 중...")
#     fomc_signal = get_current_omega('Federal Reserve interest rate')
#     tech_signal = get_current_omega('technology sector earnings')

#     print(f"FOMC 신호: 감성={fomc_signal['mean_score']:+.3f}, Ω={fomc_signal['Omega']:.2f}, 기사수={fomc_signal['volume']}")
#     print(f"기술주 신호: 감성={tech_signal['mean_score']:+.3f}, Ω={tech_signal['Omega']:.2f}, 기사수={tech_signal['volume']}")

### gdelt 구조 확인 코드

In [4]:
import gdelt

gd = gdelt.gdelt(version=2)

# Events 테이블 (하루치)
events = gd.Search(['2025 04 14'], table='events', coverage=False)
print(f"Events: {events.shape}")
display(events.head())

# Mentions 테이블
mentions = gd.Search(['2025 04 14'], table='mentions', coverage=False)
print(f"Mentions: {mentions.shape}")
display(mentions.head())

# GKG 테이블
gkg = gd.Search(['2025 04 14'], table='gkg', coverage=False)
print(f"GKG: {gkg.shape}")
display(gkg.head())

here
Events: (1076, 62)


,GLOBALEVENTID,SQLDATE,MonthYear,Year,FractionDate,Actor1Code,Actor1Name,Actor1CountryCode,Actor1KnownGroupCode,Actor1EthnicCode,...,ActionGeo_Type,ActionGeo_FullName,ActionGeo_CountryCode,ActionGeo_ADM1Code,ActionGeo_ADM2Code,ActionGeo_Lat,ActionGeo_Long,ActionGeo_FeatureID,DATEADDED,SOURCEURL
0,1237988332,20240414,202404,2024,2024.2849,CAN,CANADA,CAN,NaN,NaN,...,4,"Ottawa, Ontario, Canada",CA,CA08,12755,45.416700,-75.700000,-570760,20250414234500,https://torontosun.com/news/national/canada-ra...
1,1237988333,20240414,202404,2024,2024.2849,CAN,CANADA,CAN,NaN,NaN,...,1,"Vietnam, Republic Of",VM,VM,NaN,16.166667,107.833333,VM,20250414234500,https://torontosun.com/news/national/canada-ra...
2,1237988334,20240414,202404,2024,2024.2849,JUD,SUPREME COURT,NaN,NaN,NaN,...,4,"Montreal, Quebec, Canada",CA,CA10,12713,45.500000,-73.583300,-569541,20250414234500,https://www.cp24.com/news/canada/2025/04/14/co...
3,1237988335,20240414,202404,2024,2024.2849,JUD,SUPREME COURT,NaN,NaN,NaN,...,4,"Quebec, Quebec, Canada",CA,CA10,12750,47.500000,-72.000000,-571850,20250414234500,https://www.cp24.com/news/canada/2025/04/14/co...
4,1237988336,20240414,202404,2024,2024.2849,PTY,POLITICIAN,NaN,NaN,NaN,...,2,"Michigan, United States",US,USMI,NaN,43.350400,-84.560300,MI,20250414234500,https://www.erienewsnow.com/story/52690136/man...


Mentions: (4304, 16)


,GLOBALEVENTID,EventTimeDate,MentionTimeDate,MentionType,MentionSourceName,MentionIdentifier,SentenceID,Actor1CharOffset,Actor2CharOffset,ActionCharOffset,InRawText,Confidence,MentionDocLen,MentionDocTone,MentionDocTranslationInfo,Extras
0,1237986846,20250414233000,20250414234500,1,theguardian.com,https://www.theguardian.com/us-news/2025/apr/1...,2,275,99,154,1,100,3859,-4.881890,NaN,NaN
1,1237836301,20250414071500,20250414234500,1,prpeak.com,https://www.prpeak.com/world-news/suspect-in-a...,17,4136,-1,4191,1,30,5810,-4.766839,NaN,NaN
2,1237827877,20250414060000,20250414234500,1,prpeak.com,https://www.prpeak.com/world-news/suspect-in-a...,17,4138,4114,4132,0,20,5810,-4.766839,NaN,NaN
3,1237827879,20250414060000,20250414234500,1,prpeak.com,https://www.prpeak.com/world-news/suspect-in-a...,17,4138,4114,4193,0,20,5810,-4.766839,NaN,NaN
4,1237957924,20250414194500,20250414234500,1,greenfieldreporter.com,https://www.greenfieldreporter.com/2025/04/14/...,12,-1,3587,3606,1,50,3699,-4.838710,NaN,NaN


GKG: (1621, 27)


,GKGRECORDID,DATE,SourceCollectionIdentifier,SourceCommonName,DocumentIdentifier,Counts,V2Counts,Themes,V2Themes,Locations,...,GCAM,SharingImage,RelatedImages,SocialImageEmbeds,SocialVideoEmbeds,Quotations,AllNames,Amounts,TranslationInfo,Extras
0,20250414234500-0,20250414234500,1,politicalwire.com,https://politicalwire.com/2025/04/14/the-const...,NaN,NaN,NaN,NaN,1#United States#US#US#39.828175#-98.5795#US;1#...,...,"wc:104,c12.1:9,c12.10:10,c12.12:2,c12.13:4,c12...",NaN,NaN,NaN,NaN,NaN,"Adam Serwer,12;Supreme Court,76;Kilmar Abrego ...",NaN,NaN,<PAGE_LINKS>https://www.theatlantic.com/politi...
1,20250414234500-1,20250414234500,1,pragativadi.com,https://pragativadi.com/saudi-arabia-cuts-indi...,NaN,NaN,USPEC_POLITICS_GENERAL1;TAX_POLITICAL_PARTY;TA...,"AFFECT,1223;EPU_UNCERTAINTY,1455;TAX_ETHNICITY...","4#Jammu, Jammu And Kashmir, India#IN#IN12#32.7...",...,"wc:364,c1.4:1,c12.1:21,c12.10:24,c12.12:6,c12....",https://pragativadi.com/wp-content/uploads/202...,NaN,NaN,https://youtube.com/c/PRAGATIVADILIVE;,NaN,"Democratic Party,25;Mehbooba Mufti,54;External...","025,for Indian pilgrims,1816;",NaN,<PAGE_AUTHORS>Ananya Pattnaik</PAGE_AUTHORS><P...
2,20250414234500-2,20250414234500,1,biztoc.com,https://biztoc.com/x/c33012bb57522632,NaN,NaN,TAX_WORLDLANGUAGES;TAX_WORLDLANGUAGES_TAMARA;T...,"EPU_POLICY_POLITICAL,52;EPU_POLICY_POLITICAL,1...",NaN,...,"wc:43,c12.1:3,c12.10:1,c12.12:1,c12.3:1,c12.4:...",https://biztoc.com/cdn/c33012bb57522632_s.webp,NaN,NaN,NaN,NaN,"Amy Walter,31;Cook Political Report,60;Amy Wal...",NaN,NaN,<PAGE_PRECISEPUBTIMESTAMP>20250414222500</PAGE...
3,20250414234500-3,20250414234500,1,z94.com,https://z94.com/ixp/366/p/idle-heirs-sean-ingr...,NaN,NaN,NaN,NaN,1#Norway#NO#NO#62#10#NO,...,"wc:641,c12.1:55,c12.10:69,c12.12:27,c12.13:26,...",https://townsquare.media/site/366/files/2025/0...,NaN,NaN,https://youtube.com/@relapserecords;https://yo...,"1937|222|| took it as a challenge , but to me ...","Loudwire Nights,128;Idle Heir,201;Life Is Viol...","2,were friends,1113;",NaN,<PAGE_LINKS>https://idleheirs.bandcamp.com/;ht...
4,20250414234500-4,20250414234500,1,iheart.com,https://1360kktx.iheart.com/content/2025-04-14...,NaN,NaN,TAX_FNCACT;TAX_FNCACT_ACTOR;MEDIA_MSM;KILL;CRI...,"CRISISLEX_T11_UPDATESSYMPATHY,1528;TAX_FNCACT_...","2#New York, United States#US#USNY#42.1497#-74....",...,"wc:264,c1.2:1,c12.1:22,c12.10:22,c12.12:10,c12...",https://i.iheart.com/v3/re/new_assets/67fbb9d2...,NaN,https://pic.twitter.com/z2vmJqHjpc;https://pic...,NaN,NaN,"Actor Nicky Katt,17;Boston Public,114;Los Ange...","3,seasons of the FOX,591;",NaN,<PAGE_LINKS>https://www.iheart.com/artist/6/al...


In [9]:
gcam = gkg['GCAM']
gcam

0       wc:104,c12.1:9,c12.10:10,c12.12:2,c12.13:4,c12...
1       wc:364,c1.4:1,c12.1:21,c12.10:24,c12.12:6,c12....
2       wc:43,c12.1:3,c12.10:1,c12.12:1,c12.3:1,c12.4:...
3       wc:641,c12.1:55,c12.10:69,c12.12:27,c12.13:26,...
4       wc:264,c1.2:1,c12.1:22,c12.10:22,c12.12:10,c12...
                              ...                        
1616    wc:685,c12.1:33,c12.10:57,c12.12:9,c12.13:23,c...
1617    wc:756,c12.1:42,c12.10:88,c12.12:50,c12.13:19,...
1618    wc:389,c1.1:2,c12.1:43,c12.10:28,c12.12:1,c12....
1619    wc:739,c12.1:75,c12.10:71,c12.12:25,c12.13:23,...
1620    wc:671,c12.1:37,c12.10:44,c12.11:1,c12.12:8,c1...
Name: GCAM, Length: 1621, dtype: object

In [10]:
gkg[['Themes','V2Themes']]

,Themes,V2Themes
0,NaN,NaN
1,USPEC_POLITICS_GENERAL1;TAX_POLITICAL_PARTY;TA...,"AFFECT,1223;EPU_UNCERTAINTY,1455;TAX_ETHNICITY..."
2,TAX_WORLDLANGUAGES;TAX_WORLDLANGUAGES_TAMARA;T...,"EPU_POLICY_POLITICAL,52;EPU_POLICY_POLITICAL,1..."
3,NaN,NaN
4,TAX_FNCACT;TAX_FNCACT_ACTOR;MEDIA_MSM;KILL;CRI...,"CRISISLEX_T11_UPDATESSYMPATHY,1528;TAX_FNCACT_..."
...,...,...
1616,ELECTION;TAX_FNCACT;TAX_FNCACT_MANAGER;TAX_FNC...,"WB_678_DIGITAL_GOVERNMENT,2724;WB_694_BROADCAS..."
1617,ENV_OIL;TAX_WEAPONS;TAX_WEAPONS_BOMB;EPU_CATS_...,"TAX_FNCACT_SPOKESMAN,458;TAX_FNCACT_SPOKESMAN,..."
1618,TOURISM;TAX_FNCACT;TAX_FNCACT_TOURIST;GEN_HOLI...,"PUBLIC_TRANSPORT,2352;GEN_HOLIDAY,356;GEN_HOLI..."
1619,NaN,NaN


In [ ]:
events['IsRootEvent'].value_counts()

IsRootEvent
1    671
0    405
Name: count, dtype: int64

In [ ]:
mentions.sort_values(by='GLOBALEVENTID')

,GLOBALEVENTID,EventTimeDate,MentionTimeDate,MentionType,MentionSourceName,MentionIdentifier,SentenceID,Actor1CharOffset,Actor2CharOffset,ActionCharOffset,InRawText,Confidence,MentionDocLen,MentionDocTone,MentionDocTranslationInfo,Extras
5,1169640061,20240414001500,20250414234500,1,thepeninsulaqatar.com,http://thepeninsulaqatar.com/article/15/04/202...,10,-1,2065,2104,1,60,2729,-4.306220,NaN,NaN
17,1169640077,20240414001500,20250414234500,1,thepeninsulaqatar.com,http://thepeninsulaqatar.com/article/15/04/202...,10,2065,2097,2121,0,20,2729,-4.306220,NaN,NaN
27,1169644518,20240414004500,20250414234500,1,eco-business.com,https://www.eco-business.com/news/making-money...,8,2219,2284,2271,1,100,6855,-8.325624,NaN,NaN
15,1169645368,20240414010000,20250414234500,1,thepeninsulaqatar.com,http://thepeninsulaqatar.com/article/15/04/202...,10,2065,-1,2194,1,60,2729,-4.306220,NaN,NaN
16,1169733107,20240414133000,20250414234500,1,thepeninsulaqatar.com,http://thepeninsulaqatar.com/article/15/04/202...,10,2065,2097,2211,0,20,2729,-4.306220,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4299,1237989403,20250414234500,20250414234500,1,azfamily.com,https://www.azfamily.com/2025/04/14/fire-rips-...,5,1055,1137,1098,1,50,2391,1.007557,NaN,NaN
4300,1237989404,20250414234500,20250414234500,1,azfamily.com,https://www.azfamily.com/2025/04/14/fire-rips-...,1,211,196,145,1,50,2391,1.007557,NaN,NaN
4301,1237989405,20250414234500,20250414234500,1,niagarafallsreview.ca,https://www.niagarafallsreview.ca/news/canada/...,7,2456,2360,2367,0,40,6772,2.409639,NaN,NaN
4302,1237989406,20250414234500,20250414234500,1,bartlesvilleradio.com,https://www.bartlesvilleradio.com/pages/news/4...,1,234,-1,286,1,100,1032,-2.247191,NaN,NaN


In [ ]:
org = gkg['Organizations']

In [ ]:
def parse_v2tone(tone_str):
    """V2Tone 필드 → 7개 차원으로 분해"""
    if pd.isna(tone_str) or tone_str == '':
        return {
            'tone': None, 'pos_score': None, 'neg_score': None,
            'polarity': None, 'activity': None, 'self_ref': None, 'word_count': None
        }
    vals = tone_str.split(',')
    return {
        'tone':       float(vals[0]) if len(vals) > 0 else None,
        'pos_score':  float(vals[1]) if len(vals) > 1 else None,
        'neg_score':  float(vals[2]) if len(vals) > 2 else None,
        'polarity':   float(vals[3]) if len(vals) > 3 else None,
        'activity':   float(vals[4]) if len(vals) > 4 else None,
        'self_ref':   float(vals[5]) if len(vals) > 5 else None,
        'word_count': int(float(vals[6])) if len(vals) > 6 else None,
    }

# DataFrame에 적용
tone_df = gkg['V2Tone'].apply(parse_v2tone).apply(pd.Series)
gkg = pd.concat([gkg, tone_df], axis=1)
display(tone_df)


,tone,pos_score,neg_score,polarity,activity,self_ref,word_count
0,-1.739130,2.608696,4.347826,6.956522,21.739130,0.000000,104.0
1,-1.534527,2.301790,3.836317,6.138107,21.483376,0.000000,364.0
2,-2.040816,2.040816,4.081633,6.122449,22.448980,0.000000,43.0
3,0.280112,2.521008,2.240896,4.761905,26.330532,3.921569,641.0
4,-2.797203,2.447552,5.244755,7.692308,18.531469,0.699301,264.0
...,...,...,...,...,...,...,...
1616,-0.127877,2.046036,2.173913,4.219949,19.181586,0.639386,685.0
1617,-8.607595,1.645570,10.253165,11.898734,25.316456,0.632911,756.0
1618,3.262956,3.262956,0.000000,3.262956,17.658349,0.000000,389.0
1619,0.369914,2.836005,2.466091,5.302096,25.277435,0.493218,739.0


In [ ]:
# ============================================================
# 1-6. GDELT BigQuery 수집 (레짐 판별용)
# ============================================================
# 사전 준비:
#   uv add google-cloud-bigquery db-dtypes
#   gcloud auth application-default login
# ============================================================

from google.cloud import bigquery
import pandas as pd
import numpy as np
import os

client = bigquery.Client()

START_INT = 20160101   # EVENTS SQLDATE 형식 (YYYYMMDD)
END_INT   = 20251231
START_GKG = 20160101000000   # GKG DATE 형식 (YYYYMMDDHHMMSS)
END_GKG   = 20251231235959


# ============================================================
# Query 1 — GKG 테이블
# ============================================================
# 추출 컬럼:
#   avg_tone         : V2Tone[0] 전체 감성 평균 (범용, 교차검증용)
#   fin_pos_rate     : c6.5 (LM Positive) / wc → 긍정 단어 비율
#   fin_neg_rate     : c6.4 (LM Negative) / wc → 부정 단어 비율
#   fin_sentiment    : fin_pos_rate - fin_neg_rate → 순 감성 점수
#   fin_uncertainty  : c6.6 (LM Uncertainty) / wc → 불확실성 비율
#   article_count    : 일별 기사 수 (시장 주목도)
#
# wc(총 단어수)로 나누는 이유:
#   GCAM 값은 절대 단어 카운트 → 긴 기사일수록 숫자가 커짐
#   10단어 기사의 c6.4=2 와 1000단어 기사의 c6.4=2 는 의미가 전혀 다름
#   → wc로 나눠서 비율(0~1)로 표준화해야 기사 간 공정한 비교 가능
# ============================================================

GKG_QUERY = f"""
SELECT
    PARSE_DATE('%Y%m%d', SUBSTR(CAST(DATE AS STRING), 1, 8)) AS date,

    -- 순 금융 감성 (긍정 - 부정) / wc
    AVG(SAFE_DIVIDE(
        COALESCE(CAST(REGEXP_EXTRACT(GCAM, r'c6\\.5:([0-9]+)') AS FLOAT64), 0) -
        COALESCE(CAST(REGEXP_EXTRACT(GCAM, r'c6\\.4:([0-9]+)') AS FLOAT64), 0),
        COALESCE(CAST(REGEXP_EXTRACT(GCAM, r'wc:([0-9]+)')     AS FLOAT64), 1)
    )) AS fin_sentiment,

    COUNT(*) AS article_count

FROM `gdelt-bq.gdeltv2.gkg`
WHERE DATE BETWEEN {START_GKG} AND {END_GKG}
  AND (
        Themes LIKE '%ECON_STOCKMARKET%'
     OR Themes LIKE '%ECON_FRBRESERVE%'
     OR Themes LIKE '%ECON_INFLATION%'
     OR Themes LIKE '%ECON_UNEMPLOYMENT%'
  )
  AND SourceCommonName IN (
        'reuters.com', 'wsj.com', 'ft.com',
        'bloomberg.com', 'cnbc.com'
  )
GROUP BY date
ORDER BY date
"""



# ============================================================
# Query 2 — EVENTS 테이블
# ============================================================
# 추출 컬럼:
#   avg_shock : GoldsteinScale 평균 (-10 ~ +10)
#               이론적 근거: CAMEO 국제정치학 스케일
#               양수 = 협력/긍정 이벤트, 음수 = 갈등/충격 이벤트
#   min_shock : 일별 최솟값 = "오늘 발생한 가장 극단적 부정 충격"
#               Bear/Crisis 구분의 핵심 — 평균은 중화되어 극단값 숨김
#
# 필터 설명:
#   Actor1CountryCode = 'USA' : 미국 관련 이벤트만
#   IsRootEvent = 1           : 중복 보도 제거, 원본 이벤트만 집계
#                               (같은 사건을 100개 언론사가 보도하면
#                                IsRootEvent=0인 99개가 집계를 부풀림)
# ============================================================

EVENTS_QUERY = f"""
SELECT
    PARSE_DATE('%Y%m%d', CAST(SQLDATE AS STRING)) AS date,
    AVG(GoldsteinScale) AS avg_shock,
    MIN(GoldsteinScale) AS min_shock

FROM `gdelt-bq.gdeltv2.events`
WHERE SQLDATE BETWEEN {START_INT} AND {END_INT}
  AND Actor1CountryCode = 'USA'
  AND IsRootEvent = 1
GROUP BY date
ORDER BY date
"""


# ============================================================
# 수집 함수
# ============================================================

def fetch_gkg() -> pd.DataFrame:
    print("GKG 수집 중... (수 분 소요, BigQuery 과금 주의)")
    df = client.query(GKG_QUERY).to_dataframe()
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date').sort_index()
    print(f"  GKG 완료: {df.shape[0]:,}일 × {df.shape[1]}개 컬럼")
    return df


def fetch_events() -> pd.DataFrame:
    print("EVENTS 수집 중...")
    df = client.query(EVENTS_QUERY).to_dataframe()
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date').sort_index()
    print(f"  EVENTS 완료: {df.shape[0]:,}일 × {df.shape[1]}개 컬럼")
    return df


# ============================================================
# 수집 실행 + NYSE 캘린더 정렬
# ============================================================

gkg_df    = fetch_gkg()
events_df = fetch_events()

# 날짜 기준 병합
gdelt_raw = gkg_df.join(events_df, how='outer')

# NYSE 영업일 정렬 (Step1과 동일한 nyse_dates 사용)
# 주말/공휴일 결측 → ffill (이전 영업일 신호 유지)
nyse_dates  = pd.bdate_range(start=START, end=END, freq='B')
gdelt_daily = gdelt_raw.reindex(
    pd.date_range(start=START, end=END, freq='D')
).ffill().bfill()

df_gdelt = gdelt_daily.reindex(nyse_dates)
df_gdelt.index.name = 'Date'

print(f"\nGDELT 최종: {df_gdelt.shape}")
print(f"결측률:")
missing = df_gdelt.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "  전 컬럼 결측 0개")

df_gdelt.head()
